# 01 — Watch a journey emerge, one round at a time

Let's start outside a station, not on a train. Our invented origin `O` needs an ENTRY walk to platform `A`. The destination `Z` includes an EXIT walk from `D`.

Your goal is to explain every boarding and every clock value. These stops and times are synthetic; this notebook is not a real journey recommendation. Read [chapter 01](../docs/01_rounds_labels_and_pareto.md) alongside it.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src" / "raptor.py").is_file():
    raise RuntimeError("Start this notebook from the repository root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from src import compile_timetable, demo_timetable, raptor, WalkPolicy, parse_time, format_time
index = compile_timetable(demo_timetable())
ready = parse_time("08:00:00")
result = raptor(index, "O", "Z", ready, max_boardings=3, boarding_slack=60)
for k, row in enumerate(result.rows):
    print(f"At most {k} boardings:", {stop: format_time(label.time) for stop, label in sorted(row.items())})

## First check: what does a round count?

Round 0 permits walking. Round 1 permits one vehicle. Round 2 permits two vehicles, which means **one transfer**, not two.

The destination must be absent from round 0, reachable at 08:30 in round 1, and improved to 08:22 in round 2. The 08:21 label at `D` is not the destination arrival; EXIT still takes a minute.

In [ ]:
assert "Z" not in result.rows[0]
assert result.rows[1]["Z"].time == parse_time("08:30:00")
assert result.rows[2]["Z"].time == parse_time("08:22:00")
assert result.rows[2]["D"].time == parse_time("08:21:00")
print("Round invariant checks passed.")

## Reconstruct both useful choices

The fast option uses two boardings. The slower option avoids a transfer. Neither dominates the other on arrival time and boarding count.

Waiting is implicit between legs. `Journey.validate` checks continuity, chronology, boarding slack, the final endpoint, and the boarding count.

In [ ]:
for journey in result.journeys():
    journey.validate(boarding_slack=60)
    print(f"\nArrive {format_time(journey.arrival)} / {journey.transfers} transfers / {journey.walking_seconds}s walking")
    for leg in journey.legs:
        print(leg.kind, leg.source, "->", leg.target, format_time(leg.departure), format_time(leg.arrival), leg.trip_id or "")

## Move the ready time by one second

Entry plus boarding slack puts the first boarding threshold at exactly 08:00:00 origin readiness. Equality is legal. One second later isn't the same query.

In [ ]:
late = raptor(index, "O", "Z", ready + 1, max_boardings=3, boarding_slack=60)
assert result.journeys()[0].arrival == parse_time("08:22:00")
assert late.journeys()[0].arrival == parse_time("08:27:00")
print("At 08:00:00:", format_time(result.journeys()[0].arrival))
print("At 08:00:01:", format_time(late.journeys()[0].arrival))

## Change a hard constraint, not a ranking weight

Now explicitly permit stairs. This changes the feasible walking graph for a new request. It is not an automatic recovery path after strict mode fails.

In [ ]:
stairs_allowed = raptor(index, "O", "Z", ready, max_boardings=3, boarding_slack=60, policy=WalkPolicy(False))
assert stairs_allowed.journeys()[0].arrival == parse_time("08:21:00")
print("Strict step-free:", format_time(result.journeys()[0].arrival))
print("Stairs explicitly allowed:", format_time(stairs_allowed.journeys()[0].arrival))
print("Measured operations:", result.metrics)

## Trace the local calculation

Follow `src/raptor.py` into `affected_routes`, `scan_route`, and `close_footpaths`. Chapter 04 connects these functions to the fixture and witness checks. This notebook demonstrates the restricted local model; multi-date loading and full multicriteria state remain unimplemented.

Before moving on, explain why reading `current` instead of `previous` while boarding can hide an extra vehicle. Then run `python -m pytest -q` and find the route-order regression test.